In [1]:
import pandas as pd

df = pd.read_csv("../Data/reviews_final.csv")
print(df.shape)

(34625, 12)


In [2]:
df["negative"] = (df["sentiment"] == "negative") * 100
df["recommend"] = df["do_recommend"] * 100

In [3]:
df["short_name"] = df["product_name"]
df["n_words"] = df["clean_text"].str.split().str.len()

print(df["category"].value_counts())

category
Tablets                  17596
Echo & Smart Speakers     7262
Fire TV                   5068
Kindle                    4137
Chargers, Cables           562
Name: count, dtype: int64


In [4]:
MIN_REVIEWS = 10

def product_table(category_df):
    table = category_df.groupby("short_name").agg(
        reviews=("rating", "size"),
        avg_rating=("rating", "mean"),
        pct_negative=("negative", "mean"),
        pct_recommend=("recommend", "mean"),
    ).round(2)
    table = table[table["reviews"] >= MIN_REVIEWS]
    return table.sort_values("avg_rating", ascending=False)

product_table(df[df["category"] == "Chargers, Cables"])

,reviews,avg_rating,pct_negative,pct_recommend
short_name,,,,
amazon echo fire tv power adapter,16,4.50,6.25,NaN
amazon 5 w usb official oem charger power adapter for fire tablets kind lee readers,416,4.39,10.34,NaN
amazon 9 w power fast official oem usb charger power adapter for fire tablets kind lee readers,73,4.21,16.44,94.44
amazon fire hd 6 standing protective case 4 th generation 2014 release cayenne red,13,3.92,23.08,NaN
kindled x leather cover black fits 97 display latest 2 nd generation kindled xs,10,3.50,30.00,NaN
new amazon kindle fire hd 9 w power fast adapter charger micro usb angle cable,13,2.46,61.54,NaN


In [5]:
def complaints(category_df, product, n=2):
    unhappy = category_df[(category_df["short_name"] == product) & (category_df["rating"] <= 3) & (category_df["n_words"] >= 10)]
    examples = unhappy["clean_text"].sample(min(n, len(unhappy)), random_state=42)
    return [text[:150] for text in examples]

def pick_products(table):
    if len(table) < 2:
        return list(table.index[:3]), None
    worst = table["avg_rating"].idxmin()
    top = table.drop(worst).sort_values("reviews", ascending=False).index[:3]
    return list(top), worst

In [6]:
def product_line(table, product):
    row = table.loc[product]
    line = f"{product}: {row['avg_rating']}/5 from {row['reviews']:.0f} reviews, {row['pct_negative']}% negative"
    if pd.notna(row["pct_recommend"]):
        line += f", {row['pct_recommend']}% recommend it"
    return line

def facts_text(category):
    category_df = df[df["category"] == category]
    table = product_table(category_df)
    top, worst = pick_products(table)

    lines = [f"Category: {category} ({len(category_df)} reviews)", "", "TOP PRODUCTS (most reviewed first):"]
    for product in top:
        lines.append("- " + product_line(table, product))
        for text in complaints(category_df, product):
            lines.append(f'    complaint: "{text}"')
    lines.append("")
    if worst is None:
        lines.append("WORST PRODUCT: none")
    else:
        lines.append("WORST PRODUCT: " + product_line(table, worst))
        for text in complaints(category_df, worst):
            lines.append(f'    complaint: "{text}"')
    return "\n".join(lines)

categories = sorted(df["category"].unique())
print(categories)
print(facts_text("Chargers, Cables"))

['Chargers, Cables', 'Echo & Smart Speakers', 'Fire TV', 'Kindle', 'Tablets']
Category: Chargers, Cables (562 reviews)

TOP PRODUCTS (most reviewed first):
- amazon 5 w usb official oem charger power adapter for fire tablets kind lee readers: 4.39/5 from 416 reviews, 10.34% negative
    complaint: "a waste of money . a waste of money. you don't need this. just plug your tablet into your computer and the charge lasts for a month"
    complaint: "only lasts for less than 1 year! . this is the third power charger that i have had to buy at cost of over 30 each since my kindle fire purchase. why c"
- amazon 9 w power fast official oem usb charger power adapter for fire tablets kind lee readers: 4.21/5 from 73 reviews, 16.44% negative, 94.44% recommend it
    complaint: "defective. . worked wonderful and fast the first couple of weeks and then just stopped working. i understand that sometimes things are just a defectiv"
    complaint: "power fast how about power not! . this doesn't power any

In [7]:
import os
os.environ["HF_HOME"] = r"E:\huggingface"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

from huggingface_hub import snapshot_download
path = snapshot_download("Qwen/Qwen2.5-0.5B-Instruct")
print("Downloaded to:", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded to: E:\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct\snapshots\7ae557604adf67be50417f59c2c2f167def9a775


In [12]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
writer = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    dtype=torch.float16 if device == 0 else torch.float32,
    device=device,
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [13]:
def prompt_v1(category):
    return "Write a blog post about these products.\n\n" + facts_text(category)

def prompt_v2(category):
    category_df = df[df["category"] == category]
    top, worst = pick_products(product_table(category_df))

    template = [f"# The best and worst {category.lower()}", "[intro of two sentences]", "", "## Top picks"]
    for product in top:
        template.append(f"**{product}**: [one or two sentences: what buyers like, and how it compares with the others]")
    template += ["", "## Common complaints", "[two or three sentences: the problems unhappy buyers mention]", ""]
    if worst is not None:
        template += [f"## Product to avoid: {worst}", "[one or two sentences: why it is the weakest]", ""]
    template += ["## Verdict", "[one sentence]"]

    return (facts_text(category) + "\n\n"
            "Fill in this article template. Replace every [...] with your own text and keep everything else as it is.\n"
            "RULES: Use only the facts above. Do not invent prices, features or numbers. "
            "Do not copy the complaints, say them in your own words. "
            "If the product to avoid has a rating of 4 or more, say it is the weakest of a good group, not a bad product.\n\n"
            + "\n".join(template))

In [14]:
def write_article(prompt):
    messages = [{"role": "system", "content": "You are a writer for a product review blog."},
                {"role": "user", "content": prompt}]
    output = writer(messages, max_new_tokens=600, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

In [15]:
import re

def check_article(article, category):
    category_df = df[df["category"] == category]
    top, worst = pick_products(product_table(category_df))
    names = top + ([worst] if worst is not None else [])
    sections = ["top picks", "common complaints", "verdict"]
    numbers_in_facts = set(re.findall(r"\d+(?:\.\d+)?", facts_text(category)))
    numbers_in_article = set(re.findall(r"\d+(?:\.\d+)?", article.replace(",", "")))
    return {
        "words": len(article.split()),
        "sections": f"{sum(section in article.lower() for section in sections)}/{len(sections)}",
        "product names": f"{sum(name in article for name in names)}/{len(names)}",
        "invented numbers": sorted(numbers_in_article - numbers_in_facts),
    }

In [16]:
for name, prompt_function in {"v1": prompt_v1, "v2": prompt_v2}.items():
    article = write_article(prompt_function("Kindle"))
    print(f"\n========== {name}", check_article(article, "Kindle"))
    print(article)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=600)


========== v1 {'words': 481, 'sections': '0/3', 'product names': '0/4', 'invented numbers': []}
### Top Products Review: Amazon Kindle Paper White E-reader

#### Product Overview:
The Amazon Kindle Paper White E-reader is a popular choice among avid readers who appreciate its sleek design and convenient touch-screen interface. This model comes in two primary colors—white and black—and has been praised for its durability and user-friendly features.

#### Product Reviews:

**Amazon Kindle Paper White E-reader 4 GB 6 Monochrome Paper White Touchscreen Wi-Fi Black**
- **Pros:** The Kindle Paper White is known for its excellent battery life and long-lasting battery life, making it ideal for those who need a reliable device for extended periods.
- **Cons:** Some users find the paper texture on the pages to be slightly uncomfortable, especially during longer readings.
- **Recommendation:** Highly recommended for those looking for a durable and comfortable e-reader that can handle regular use

In [17]:
articles = []
for category in categories:
    article = write_article(prompt_v2(category))
    print(category, check_article(article, category))
    articles.append(article)

with open("../Reports/articles.md", "w") as file:
    file.write("\n\n---\n\n".join(articles))
print("saved", len(articles), "articles")

[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chargers, Cables {'words': 191, 'sections': '2/3', 'product names': '0/4', 'invented numbers': []}


[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Echo & Smart Speakers {'words': 402, 'sections': '3/3', 'product names': '0/2', 'invented numbers': []}


[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fire TV {'words': 190, 'sections': '3/3', 'product names': '0/1', 'invented numbers': []}
Kindle {'words': 364, 'sections': '3/3', 'product names': '0/4', 'invented numbers': []}


[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Tablets {'words': 336, 'sections': '3/3', 'product names': '0/4', 'invented numbers': []}
saved 5 articles
